In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [5]:
from src.data.preprocess import TextPreprocessor

In [6]:
with open("../data/raw/wikitext.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs


In [7]:
processor = TextPreprocessor()

clean_text = processor.clean_text(text)

print(clean_text[:500])

= valkyria chronicles iii = senjō no valkyria 3 : unrecorded chronicles ( japanese : 戦場のヴァルキュリア3 , lit . valkyria of the battlefield 3 ) , commonly referred to as valkyria chronicles iii outside japan , is a tactical role @-@ playing video game developed by sega and media.vision for the playstation portable . released in january 2011 in japan , it is the third game in the valkyria series . employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs paral


In [8]:
words = clean_text.split()

print("Total Words:", len(words))
print("Unique Words:", len(set(words)))

Total Words: 2051910
Unique Words: 66649


In [9]:
from src.data.vocab import Vocabulary

In [10]:
words = clean_text.split()

print("Total Words:", len(words))

Total Words: 2051910


In [11]:
vocab = Vocabulary(min_freq=2)

vocab.build_vocab(words)

print("Vocabulary Size:", len(vocab.word_to_idx))

Vocabulary Size: 39275


In [10]:
list(vocab.word_to_idx.items())[:20]

[('<PAD>', 0),
 ('<UNK>', 1),
 ('=', 2),
 ('valkyria', 3),
 ('chronicles', 4),
 ('iii', 5),
 ('senjō', 6),
 ('no', 7),
 ('3', 8),
 (':', 9),
 ('unrecorded', 10),
 ('(', 11),
 ('japanese', 12),
 ('戦場のヴァルキュリア3', 13),
 (',', 14),
 ('lit', 15),
 ('.', 16),
 ('of', 17),
 ('the', 18),
 ('battlefield', 19)]

In [11]:
list(vocab.idx_to_word.items())[:20]

[(0, '<PAD>'),
 (1, '<UNK>'),
 (2, '='),
 (3, 'valkyria'),
 (4, 'chronicles'),
 (5, 'iii'),
 (6, 'senjō'),
 (7, 'no'),
 (8, '3'),
 (9, ':'),
 (10, 'unrecorded'),
 (11, '('),
 (12, 'japanese'),
 (13, '戦場のヴァルキュリア3'),
 (14, ','),
 (15, 'lit'),
 (16, '.'),
 (17, 'of'),
 (18, 'the'),
 (19, 'battlefield')]

In [12]:
word_ids = vocab.numericalize(words)

print(word_ids[:20])

[2, 3, 4, 5, 2, 6, 7, 3, 8, 9, 10, 4, 11, 12, 9, 13, 14, 15, 16, 3]


In [13]:
for idx in word_ids[:10]:
    print(idx, "->", vocab.idx_to_word[idx])

2 -> =
3 -> valkyria
4 -> chronicles
5 -> iii
2 -> =
6 -> senjō
7 -> no
3 -> valkyria
8 -> 3
9 -> :


In [14]:
from src.data.sequence_generator import SequenceGenerator

In [15]:
generator = SequenceGenerator(
    sequence_length=10
)

X, y = generator.generate(word_ids)

In [16]:
print("Total Samples:", len(X))

print("First Input:", X[0])

print("First Target:", y[0])

Total Samples: 2051900
First Input: [2, 3, 4, 5, 2, 6, 7, 3, 8, 9]
First Target: 10


In [17]:
decoded = [
    vocab.idx_to_word[idx]
    for idx in X[0]
]

print(decoded)

print(
    "Target:",
    vocab.idx_to_word[y[0]]
)

['=', 'valkyria', 'chronicles', 'iii', '=', 'senjō', 'no', 'valkyria', '3', ':']
Target: unrecorded


In [18]:
print(f"Samples: {len(X):,}")

Samples: 2,051,900


In [19]:
MAX_SAMPLES = 200000

X = X[:MAX_SAMPLES]
y = y[:MAX_SAMPLES]

print(len(X))
print(len(y))

200000
200000


In [20]:
from src.data.dataset import NextWordDataset

In [21]:
dataset = NextWordDataset(X, y)

print(len(dataset))

200000


In [22]:
sample_x, sample_y = dataset[0]

print(sample_x)
print(sample_y)

tensor([2, 3, 4, 5, 2, 6, 7, 3, 8, 9])
tensor(10)


In [23]:
from torch.utils.data import DataLoader

In [24]:
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [25]:
batch_x, batch_y = next(iter(train_loader))

print(batch_x.shape)
print(batch_y.shape)

torch.Size([32, 10])
torch.Size([32])


In [26]:
import pickle

with open("../data/processed/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open("../data/processed/X.pkl", "wb") as f:
    pickle.dump(X, f)

with open("../data/processed/y.pkl", "wb") as f:
    pickle.dump(y, f)